# One gNodeB, Two Fixed UEs: 100-Step Radio Trace

This notebook creates one fixed gNodeB and two fixed eMBB UEs, runs 100 simulator steps, and records radio, queue, service, and PRB-load data.

Goal: check whether load is always high, or whether it follows real queue/demand and scheduler allocation.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scenario_creator import create_multignb_env

plt.rcParams.update({"figure.dpi": 125, "axes.grid": True, "grid.alpha": 0.25})

## Scenario

- One gNodeB with `150` total PRBs.
- Equal slice budget: `50 eMBB`, `50 URLLC`, `50 mMTC`.
- Two fixed eMBB UEs near the gNodeB.
- `max_prbs_per_ue = 20`, so the two UEs can use at most `40/50 = 0.8` eMBB slice load.
- Traffic schedule creates changing demand so the load should not be permanently high if queues drain.

In [ ]:
SEED = 77
N_STEPS = 100
SLICE_TYPE = "eMBB"
TOTAL_PRBS = 150
SLICE_PRB_BUDGETS = {"eMBB": 50, "URLLC": 50, "mMTC": 50}
MAX_PRBS_PER_UE = 20

rng = np.random.default_rng(SEED)

env = create_multignb_env(
    rng=rng,
    n=4,
    n_gnbs=1,
    gnb_configs=[{
        "id": 0,
        "x": 0.0,
        "y": 0.0,
        "coverage_radius": 500.0,
        "carrier_id": 0,
        "bandwidth_hz": 20e6,
        "n_prbs": TOTAL_PRBS,
    }],
    slots_per_step=5,
    L1_level=False,
    step_dt=1e-3,
    mobility_dt=0.0,
    radio_substeps=1,
    max_episode_steps=N_STEPS + 5,
    slice_prb_budgets=SLICE_PRB_BUDGETS,
    max_prbs_per_ue=MAX_PRBS_PER_UE,
)

obs, info = env.reset(seed=SEED)

ue_ids = []
ue_ids.append(env.add_ue(
    x=35.0,
    y=0.0,
    vx=0.0,
    vy=0.0,
    slice_type=SLICE_TYPE,
    bit_rate=300_000.0,
    packet_size_bits=600.0,
    bit_rate_schedule=[
        {"step": 25, "bit_rate": 2_000_000.0},
        {"step": 55, "bit_rate": 0.0},
        {"step": 75, "bit_rate": 600_000.0},
    ],
))
ue_ids.append(env.add_ue(
    x=80.0,
    y=40.0,
    vx=0.0,
    vy=0.0,
    slice_type=SLICE_TYPE,
    bit_rate=600_000.0,
    packet_size_bits=600.0,
    bit_rate_schedule=[
        {"step": 25, "bit_rate": 3_000_000.0},
        {"step": 55, "bit_rate": 0.0},
        {"step": 75, "bit_rate": 800_000.0},
    ],
))

print("UE ids:", ue_ids)
print("eMBB budget PRBs:", env.get_slice_prb_budget(0, SLICE_TYPE))
print("max PRBs per UE:", env.max_prbs_per_ue)
for ue_id in ue_ids:
    print("UE", ue_id, "serving gNB", env.get_ue(ue_id).serving_gnb)

## Run 100 Steps And Record Everything

In [ ]:
slice_rows = []
ue_rows = []

for step in range(N_STEPS):
    obs, reward, terminated, truncated, info = env.step(0)
    kpi = info["slice_kpis"][(0, SLICE_TYPE)]

    slice_rows.append({
        "step": step,
        "budget_prbs": kpi["budget_prbs"],
        "allocated_prbs": kpi["allocated_prbs"],
        "used_prbs": kpi["used_prbs"],
        "allocated_load": kpi["allocated_load"],
        "demand_prbs": kpi["demand_prbs"],
        "demand_load": kpi["demand_load"],
        "queue_bits": kpi["queue_bits"],
        "queue_pressure": kpi["queue_pressure"],
        "arrived_bits": kpi["arrived_bits"],
        "scheduled_bits": kpi["scheduled_bits"],
        "served_bits": kpi["served_bits"],
        "served_ratio": kpi["served_ratio"],
        "service_to_arrival_ratio": kpi["service_to_arrival_ratio"],
        "tx_success_ratio": kpi["tx_success_ratio"],
        "ue_count": kpi["ue_count"],
        "throughput_bps": kpi["throughput_bps"],
        "delay_s": kpi["delay_s"],
        "dropped_bits": kpi["dropped_bits"],
    })

    for ue_id in ue_ids:
        radio = env.get_ue_radio_metrics(ue_id)
        radio["step"] = step
        radio["ue_id"] = ue_id
        ue_rows.append(radio)

    if terminated or truncated:
        break

slice_trace = pd.DataFrame(slice_rows)
ue_trace = pd.DataFrame(ue_rows)

display(slice_trace.head(10))
display(slice_trace.tail(10))
display(ue_trace.head(10))

## Summary: Is Load Always Up?

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "steps",
        "budget_prbs",
        "max_allocated_prbs",
        "mean_allocated_prbs",
        "steps_allocated_zero",
        "steps_allocated_less_than_budget",
        "max_allocated_load",
        "mean_allocated_load",
        "max_demand_load",
        "max_queue_pressure",
    ],
    "value": [
        len(slice_trace),
        int(slice_trace["budget_prbs"].iloc[0]),
        float(slice_trace["allocated_prbs"].max()),
        float(slice_trace["allocated_prbs"].mean()),
        int((slice_trace["allocated_prbs"] == 0).sum()),
        int((slice_trace["allocated_prbs"] < slice_trace["budget_prbs"]).sum()),
        float(slice_trace["allocated_load"].max()),
        float(slice_trace["allocated_load"].mean()),
        float(slice_trace["demand_load"].max()),
        float(slice_trace["queue_pressure"].max()),
    ],
})
display(summary)

assert (slice_trace["allocated_prbs"] <= slice_trace["budget_prbs"]).all()
assert (slice_trace["allocated_prbs"] <= len(ue_ids) * MAX_PRBS_PER_UE).all()

## Plots

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(11, 13), sharex=True, constrained_layout=True)

axes[0].plot(slice_trace["step"], slice_trace["budget_prbs"], label="budget_prbs", linewidth=2)
axes[0].plot(slice_trace["step"], slice_trace["allocated_prbs"], label="allocated/used_prbs", linewidth=2)
axes[0].plot(slice_trace["step"], slice_trace["demand_prbs"], label="demand_prbs", linewidth=2)
axes[0].axhline(len(ue_ids) * MAX_PRBS_PER_UE, color="tab:red", linestyle="--", label="2 UE cap")
axes[0].set_ylabel("PRBs")
axes[0].legend(loc="upper left")

axes[1].plot(slice_trace["step"], slice_trace["allocated_load"], label="allocated_load")
axes[1].plot(slice_trace["step"], slice_trace["demand_load"], label="demand_load")
axes[1].set_ylabel("load")
axes[1].legend(loc="upper left")

axes[2].plot(slice_trace["step"], slice_trace["queue_bits"], label="queue_bits")
axes[2].plot(slice_trace["step"], slice_trace["arrived_bits"], label="arrived_bits")
axes[2].plot(slice_trace["step"], slice_trace["served_bits"], label="served_bits")
axes[2].set_ylabel("bits")
axes[2].legend(loc="upper left")

for ue_id in ue_ids:
    sample = ue_trace[ue_trace["ue_id"] == ue_id]
    axes[3].plot(sample["step"], sample["allocated_prbs"], label=f"UE {ue_id} allocated_prbs")
axes[3].set_ylabel("UE PRBs")
axes[3].legend(loc="upper left")

for ue_id in ue_ids:
    sample = ue_trace[ue_trace["ue_id"] == ue_id]
    axes[4].plot(sample["step"], sample["rx_probability"], label=f"UE {ue_id} rx_probability")
axes[4].plot(slice_trace["step"], slice_trace["tx_success_ratio"], label="slice tx_success_ratio", color="black", linestyle="--")
axes[4].set_ylabel("ratio")
axes[4].set_xlabel("step")
axes[4].legend(loc="upper left")

plt.show()

## Save CSVs

In [ ]:
out_dir = PROJECT_ROOT / "results" / "one_gnb_two_fixed_ues_radio_trace"
out_dir.mkdir(parents=True, exist_ok=True)
slice_csv = out_dir / "slice_trace.csv"
ue_csv = out_dir / "ue_radio_trace.csv"
slice_trace.to_csv(slice_csv, index=False)
ue_trace.to_csv(ue_csv, index=False)
print("Saved", slice_csv)
print("Saved", ue_csv)

## Close Environment

In [ ]:
env.close()